# training.ipynb

Notebook ini adalah **final training notebook** untuk LPSE-X pada benchmark data riil multi-tahun. Tujuannya adalah menunjukkan bahwa model akhir dapat diretrain dari artefak split-aware yang sudah dibekukan, lalu diekspor kembali ke format yang dipakai pada inference.

Notebook ini sengaja dibuat ringkas dan fokus pada jalur training akhir. Proses yang lebih besar seperti pembuatan raw split, materialisasi fitur, dan diagnostik tambahan tetap berada di pipeline repo (`src/` dan `scripts/`).

## A. Ringkasan Artefak dan Prasyarat

Sebelum training dijalankan, notebook ini membaca artefak inti berikut:

- `train_data/features.parquet` dan `train_data/labels.parquet`
- `test_data/features.parquet` dan `test_data/labels.parquet`
- `models/best_params.json`
- `data/processed/feature_manifest.json`
- `data/processed/split_metadata.json`
- `models/calibration.json`

Tujuannya adalah memastikan reviewer dapat langsung melihat **skala data, jumlah fitur, batas split temporal, dan konfigurasi training** yang digunakan pada model final.

In [1]:
from pathlib import Path
import json

feature_manifest = json.loads(Path("data/processed/feature_manifest.json").read_text())
split_metadata = json.loads(Path("data/processed/split_metadata.json").read_text())
calibration = json.loads(Path("models/calibration.json").read_text())
best_params = json.loads(Path("models/best_params.json").read_text())

print({
    "train_rows": split_metadata["train_count"],
    "test_rows": split_metadata["test_count"],
    "split_date": split_metadata["split_date"],
    "feature_count": feature_manifest["feature_count"],
    "calibration_enabled": calibration["enabled"],
    "calibration_samples": calibration["n_calibration_samples"],
    "n_rounds": best_params["n_rounds"],
})

{'train_rows': 372150, 'test_rows': 93034, 'split_date': '2023-03-10 07:27:51+00:00', 'feature_count': 34, 'calibration_enabled': True, 'calibration_samples': 287, 'n_rounds': 321}


## B. Memuat Data Train/Test yang Sudah Dipisah

Cell berikut memuat fitur dan label dari artefak yang sudah dibekukan. Dengan pendekatan ini, notebook menunjukkan jalur training yang **reproducible** sekaligus tetap menjaga disiplin pemisahan train/test.

In [2]:
from pathlib import Path
import json
import pandas as pd
import xgboost as xgb
from src.model import compute_sample_weights

train_X = pd.read_parquet("train_data/features.parquet")
train_y = pd.read_parquet("train_data/labels.parquet")["risk_label"]
test_X = pd.read_parquet("test_data/features.parquet")
test_y = pd.read_parquet("test_data/labels.parquet")["risk_label"]
print(train_X.shape, test_X.shape)


(372150, 34) (93034, 34)


## C. Retraining Model Final

Cell ini melatih ulang model XGBoost final menggunakan parameter terbaik yang sudah dikunci. Output utama dari tahap ini adalah `models/xgb_model.ubj`, yaitu artefak model yang dipakai pada notebook inference dan jalur deployment lokal.

In [3]:
params = json.loads(Path("models/best_params.json").read_text())
n_rounds = int(params.pop("n_rounds", 449))
weights = compute_sample_weights(train_y)
model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    seed=42,
    n_estimators=n_rounds,
    n_jobs=-1,
    **params,
)
model.fit(train_X, train_y, sample_weight=weights)
model.save_model("models/xgb_model.ubj")
print("saved", Path("models/xgb_model.ubj").exists())


saved True


## D. Evaluasi pada Held-Out Test Split

Evaluasi dilakukan terhadap test split yang sudah dipisahkan sebelumnya. Bagian ini menampilkan metrik ringkas dan classification report agar pembaca bisa langsung melihat kualitas model akhir.

In [4]:
from sklearn.metrics import classification_report, f1_score, accuracy_score
preds = model.predict(test_X)
print({
    "accuracy": round(float(accuracy_score(test_y, preds)), 4),
    "macro_f1": round(float(f1_score(test_y, preds, average="macro")), 4),
    "weighted_f1": round(float(f1_score(test_y, preds, average="weighted")), 4),
})
print(classification_report(test_y, preds))


{'accuracy': 0.9899, 'macro_f1': 0.9831, 'weighted_f1': 0.9899}
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     26358
           1       0.99      0.99      0.99     58425
           2       0.99      0.94      0.96      8251

    accuracy                           0.99     93034
   macro avg       0.99      0.98      0.98     93034
weighted avg       0.99      0.99      0.99     93034



## E. Ekspor Model ONNX

Selain artefak `.ubj`, notebook juga mengekspor model ke format `ONNX` agar jalur inference lokal memiliki opsi deployment yang lebih ringan dan interoperable.

In [5]:
renamed = train_X.copy()
renamed.columns = [f"f{i}" for i in range(renamed.shape[1])]
onnx_model_src = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    seed=42,
    n_estimators=n_rounds,
    n_jobs=-1,
    **params,
)
onnx_model_src.fit(renamed, train_y, sample_weight=weights)
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType
initial_type = [("float_input", FloatTensorType([None, renamed.shape[1]]))]
onnx_model = convert_xgboost(onnx_model_src, initial_types=initial_type)
with open("models/xgb_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())
print("saved", Path("models/xgb_model.onnx").exists())


saved True
